You could use this notebook on [Google Colab](https://colab.research.google.com/) with a GPU Hardware Accelerator runtype.

# Setup


In [1]:
#For some reason pip3 install -r requirements.txt doesn't work on Colab
! pip3 install -r requirements.txt


[notice] A new release of pip available: 22.2.1 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# Exploratory Data Analysis
A key to the usefulness of this program is understanding the dataset. Knowing the limitations of the current data adds context to the predictions made by the program.

In [20]:
# Import necessary libraries
import pandas as pd
import numpy as np
from nba_api.stats.endpoints import teamgamelog, cumestatsteamgames, cumestatsteam, gamerotation, boxscoreadvancedv3, leaguegamefinder
from nba_api.stats.static import teams, players
import matplotlib.pyplot as plt
import seaborn as sns
import json

pd.option_context('mode.chained_assignment', None)
pd.options.display.max_columns = None
pd.options.display.max_rows = None

In [3]:
nba_teams = teams.get_teams()
print(f"Number of teams fetched: {len(nba_teams)}")
celtics = [team for team in nba_teams if team['abbreviation'] == 'BOS'][0]
celtics_id = celtics['id']

# query for games where the celtics were playing
gamefinder = leaguegamefinder.LeagueGameFinder(team_id_nullable=celtics_id)
games = gamefinder.get_data_frames()[0]
# Will only give stats for the Celtics, not the opponent
games.columns


Number of teams fetched: 30


Index(['SEASON_ID', 'TEAM_ID', 'TEAM_ABBREVIATION', 'TEAM_NAME', 'GAME_ID',
       'GAME_DATE', 'MATCHUP', 'WL', 'MIN', 'PTS', 'FGM', 'FGA', 'FG_PCT',
       'FG3M', 'FG3A', 'FG3_PCT', 'FTM', 'FTA', 'FT_PCT', 'OREB', 'DREB',
       'REB', 'AST', 'STL', 'BLK', 'TOV', 'PF', 'PLUS_MINUS'],
      dtype='object')

In [4]:
games.groupby(games.SEASON_ID.str[-4:])[['GAME_ID']].count()

,GAME_ID
SEASON_ID,
1983,105
1984,103
1985,100
1986,105
1987,99
1988,85
1989,87
1990,93
1991,92


In [8]:
# get all the games
result = leaguegamefinder.LeagueGameFinder(league_id_nullable='00', season_type_nullable='Regular Season')
all_games = result.get_data_frames()[0]
# Example game
full_game = all_games[all_games.GAME_ID == '0022401058']
full_game

,SEASON_ID,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,PTS,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,REB,AST,STL,BLK,TOV,PF,PLUS_MINUS
31,22024,1610612756,PHX,Phoenix Suns,0022401058,2025-03-26,PHX vs. BOS,L,239,102,36,87,0.414,13,41,0.317,17,21,0.810,11,27,38,22,7,3,13,18,-30.0
39,22024,1610612738,BOS,Boston Celtics,0022401058,2025-03-26,BOS @ PHX,W,239,132,45,89,0.506,22,52,0.423,20,22,0.909,11,37,48,33,8,7,10,18,30.0


In [13]:
# sort games by date
all_games['GAME_DATE'] = pd.to_datetime(all_games['GAME_DATE'], format='%Y-%m-%d')
sorted_games = all_games.sort_values(['TEAM_ID', 'GAME_DATE'], ascending=True)
print(sorted_games.tail(5))

    SEASON_ID     TEAM_ID TEAM_ABBREVIATION          TEAM_NAME     GAME_ID  \
126     22024  1610612766               CHA  Charlotte Hornets  0022401010   
113     22024  1610612766               CHA  Charlotte Hornets  0022401017   
85      22024  1610612766               CHA  Charlotte Hornets  0022401032   
50      22024  1610612766               CHA  Charlotte Hornets  0022401045   
0       22024  1610612766               CHA  Charlotte Hornets  0022401069   

     GAME_DATE      MATCHUP WL  MIN  PTS  FGM  FGA  FG_PCT  FG3M  FG3A  \
126 2025-03-20  CHA vs. NYK  W  239  115   43   88   0.489    15    32   
113 2025-03-21    CHA @ OKC  L  241  106   37   80   0.463    19    41   
85  2025-03-23    CHA @ MIA  L  239  105   41   87   0.471    11    33   
50  2025-03-25  CHA vs. ORL  L  241  104   37   88   0.420    14    34   
0   2025-03-28    CHA @ TOR  L  239   97   39   96   0.406     9    31   

     FG3_PCT  FTM  FTA  FT_PCT  OREB  DREB  REB  AST  STL  BLK  TOV  PF  \
126    0.46

In [29]:
season_2024 = sorted_games[sorted_games['SEASON_ID']=='22024']

# Create columns representing the rolling average of the previous 5 games
features = ['PTS', 'FGM', 'FG_PCT', 'FG3M', 'FG3_PCT', 'FTM', 'FT_PCT', 'OREB', 'DREB', 'AST', 'STL', 'BLK', 'TOV', 'PF']

teams = season_2024['TEAM_ABBREVIATION'].unique()
for team in teams:
    for feature in features:
        season_2024.loc[season_2024['TEAM_ABBREVIATION']==team, [f'rolling_{feature}']] = season_2024[season_2024['TEAM_ABBREVIATION']==team].shift(1).rolling(window=5, min_periods=0)[feature].mean()
season_2024.head(5)

,SEASON_ID,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,PTS,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,REB,AST,STL,BLK,TOV,PF,PLUS_MINUS,rolling_PTS,rolling_FGM,rolling_FG_PCT,rolling_FG3M,rolling_FG3_PCT,rolling_FTM,rolling_FT_PCT,rolling_OREB,rolling_DREB,rolling_AST,rolling_STL,rolling_BLK,rolling_TOV,rolling_PF
2180,22024,1610612737,ATL,Atlanta Hawks,0022400064,2024-10-23,ATL vs. BKN,W,241,120,39,80,0.488,9,28,0.321,33,46,0.717,12,33,45,25,12,9,16,20,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2152,22024,1610612737,ATL,Atlanta Hawks,0022400079,2024-10-25,ATL vs. CHA,W,240,125,39,81,0.481,14,38,0.368,33,38,0.868,7,32,39,25,11,9,13,28,5.0,120.000000,39.00,0.4880,9.0,0.321000,33.000000,0.717000,12.0,33.000000,25.000000,12.0,9.000000,16.0,20.000000
2130,22024,1610612737,ATL,Atlanta Hawks,0022400100,2024-10-27,ATL @ OKC,L,240,104,36,91,0.396,10,31,0.323,22,29,0.759,17,32,49,24,7,4,19,23,-24.0,122.500000,39.00,0.4845,11.5,0.344500,33.000000,0.792500,9.5,32.500000,25.000000,11.5,9.000000,14.5,24.000000
2114,22024,1610612737,ATL,Atlanta Hawks,0022400103,2024-10-28,ATL vs. WAS,L,240,119,39,81,0.481,15,40,0.375,26,36,0.722,6,33,39,32,12,7,16,22,-2.0,116.333333,38.00,0.4550,11.0,0.337333,29.333333,0.781333,12.0,32.333333,24.666667,10.0,7.333333,16.0,23.666667
2083,22024,1610612737,ATL,Atlanta Hawks,0022400121,2024-10-30,ATL @ WAS,L,240,120,45,95,0.474,12,39,0.308,18,21,0.857,12,29,41,28,10,3,15,19,-13.0,117.000000,38.25,0.4615,12.0,0.346750,28.500000,0.766500,10.5,32.500000,26.500000,10.5,7.250000,16.0,23.250000


In [30]:
season_2024[season_2024['TEAM_ABBREVIATION']=='BOS']

,SEASON_ID,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,PTS,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,REB,AST,STL,BLK,TOV,PF,PLUS_MINUS,rolling_PTS,rolling_FGM,rolling_FG_PCT,rolling_FG3M,rolling_FG3_PCT,rolling_FTM,rolling_FT_PCT,rolling_OREB,rolling_DREB,rolling_AST,rolling_STL,rolling_BLK,rolling_TOV,rolling_PF
2201,22024,1610612738,BOS,Boston Celtics,0022400061,2024-10-22,BOS vs. NYK,W,240,132,48,95,0.505,29,61,0.475,7,8,0.875,11,29,40,33,6,3,3,15,23.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2179,22024,1610612738,BOS,Boston Celtics,0022400073,2024-10-24,BOS @ WAS,W,240,122,42,90,0.467,17,45,0.378,21,27,0.778,12,37,49,21,7,3,14,19,20.0,132.00,48.000000,0.505000,29.000000,0.4750,7.00,0.87500,11.0,29.0,33.000000,6.000000,3.00,3.00,15.000000
2151,22024,1610612738,BOS,Boston Celtics,0022400089,2024-10-26,BOS @ DET,W,239,124,38,90,0.422,22,48,0.458,26,30,0.867,13,24,37,19,10,6,10,16,6.0,127.00,45.000000,0.486000,23.000000,0.4265,14.00,0.82650,11.5,33.0,27.000000,6.500000,3.00,8.50,17.000000
2106,22024,1610612738,BOS,Boston Celtics,0022400104,2024-10-28,BOS vs. MIL,W,242,119,43,87,0.494,18,47,0.383,15,19,0.789,4,34,38,24,5,5,12,18,11.0,126.00,42.666667,0.464667,22.666667,0.4370,18.00,0.84000,12.0,30.0,24.333333,7.666667,4.00,9.00,16.666667
2086,22024,1610612738,BOS,Boston Celtics,0022400119,2024-10-30,BOS @ IND,L,266,132,44,109,0.404,19,57,0.333,25,31,0.806,18,33,51,23,11,5,14,19,-3.0,124.25,42.750000,0.472000,21.500000,0.4235,17.25,0.82725,10.0,31.0,24.250000,7.000000,4.25,9.75,17.000000
2061,22024,1610612738,BOS,Boston Celtics,0022400132,2024-11-01,BOS @ CHA,W,242,124,41,86,0.477,13,42,0.310,29,35,0.829,11,30,41,21,12,5,10,13,15.0,125.80,43.000000,0.458400,21.000000,0.4054,18.80,0.82300,11.6,31.4,24.000000,7.800000,4.40,10.60,17.400000
2025,22024,1610612738,BOS,Boston Celtics,0022400141,2024-11-02,BOS @ CHA,W,240,113,36,82,0.439,16,52,0.308,25,28,0.893,8,36,44,23,4,4,15,17,10.0,124.20,41.600000,0.452800,17.800000,0.3724,23.20,0.81380,11.6,31.6,21.600000,9.000000,4.80,12.00,17.000000
1995,22024,1610612738,BOS,Boston Celtics,0022400157,2024-11-04,BOS @ ATL,W,239,123,48,99,0.485,18,55,0.327,9,13,0.692,11,39,50,27,15,7,12,14,30.0,122.40,40.400000,0.447200,17.600000,0.3584,24.00,0.83680,10.8,31.4,22.000000,8.400000,5.00,12.20,16.600000
1969,22024,1610612738,BOS,Boston Celtics,0022400172,2024-11-06,BOS vs. GSW,L,240,112,38,90,0.422,19,54,0.352,17,21,0.810,13,34,47,22,5,3,12,18,-6.0,122.20,42.400000,0.459800,16.800000,0.3322,20.60,0.80180,10.4,34.4,23.600000,9.400000,5.20,12.60,16.200000
1950,22024,1610612738,BOS,Boston Celtics,0022400187,2024-11-08,BOS vs. BKN,W,266,108,39,90,0.433,14,53,0.264,16,19,0.842,4,35,39,23,7,9,13,15,4.0,120.80,41.400000,0.445400,17.000000,0.3260,21.00,0.80600,12.2,34.4,23.200000,9.400000,4.80,12.60,16.200000


## Feature Definitions

Each row represents a matchup between two teams.  
TEAM_NAME: Name of the home team. All team statistics count that season's games up to but not including the current game
Days-Rest-Home: The difference in days between the current game and the home team's previous game
GP: games played by the home team
W: wins by the home team
L: losses by the home team
W_PCT: win percentage of the home team
MIN: total minutes  
FGM: Made field goals  
FGA: attempted field goals  
FG_PCT: percentage of field goals made  
FG3M: three point field goals made  
FG3A: three point field goals attempted  
FG3_PCT: percentage of three point field goals made  
FTM: free throws made  
FTA: free throws attempted   
FT_PCT: percentage of free throws made  
OREB: offensive rebounds  
DREB: defensive rebounds  
REB: total rebounds  
AST: assists  
TOV: turnovers  
STL: steals  
BLK: blocks  
BLKA: blocks against  
PF: personal fouls   
PFD: personal fouls drawn  
PTS: total points  
PLUS_MINUS: point differential   
TEAM_NAME.1: Name of the away team. The away team's statistics are represented with a '.1' suffix  
Days-Rest-Away: Days of rest since the away team's previous game
Date(str): Date of the game in format '2012-11-04'  
Home-Team-Win(float): A boolean representing whether the home team won the game  
Score(float): The total score for the game  
OU(float): The predicted over-under value by the sportsbook before the game  
OU-Cover(float): Whether the game's total score exceeded the over under  

# TODO:
Where is the over under coming from for these games before gambling was legal?  
Clean data: convert to ints, remove highly correlated features, remove ranks  
minutes: include to normalize overtime games  
points: normalize based on average for a season
data warehousing: store the data in a way so that it is easier to run high level queries eg. get all of the Lakers games for the 2024-2025 season: model for team, game, and season
The goal is to create sequences of the previous 5 games for each team including statistics for and statistics allowed
